<a href="https://colab.research.google.com/github/gunavathibaskaran170/AI-project-/blob/main/Titanic_survived_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_csv("/content/drive/MyDrive/DATASET /titanic.csv")
df.head(30)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


# **Applyinh simpleImputer to handle missing Values**

In [4]:
from sklearn.impute import SimpleImputer
i1=SimpleImputer(strategy='mean')
i2=SimpleImputer(strategy='most_frequent')
df[['Age']]=i1.fit_transform(df[['Age']])
df[['Embarked']]=i2.fit_transform(df[['Embarked']])

In [7]:
df = df.drop(['Cabin', 'Ticket'], axis=1)

In [8]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


# EDA

In [ ]:
for i in df.columns:
  plt.figure(figsize=(4,6))
  sns.boxplot(df[i])

# **Detect the outlier**

In [48]:
from sklearn.ensemble import IsolationForest
iso=IsolationForest(contamination=0.1)
df['outlier']=iso.fit_predict(df[['SibSp','Parch','Fare']])
df=df[df['outlier']==-1]

In [49]:
df = df.drop(columns='outlier')

In [50]:
from scipy.stats.mstats import winsorize
df['Fare']=winsorize(df['Fare'],limits=[0.05,0.05])

In [51]:
from sklearn.preprocessing import LabelEncoder

lc = LabelEncoder()
for col in ["Sex", "Embarked"]:
    df[col] = lc.fit_transform(df[col])


In [70]:
X=df[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]
y=df['Survived']

In [71]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [126]:
from sklearn.ensemble import RandomForestClassifier
model =RandomForestClassifier(n_estimators=500,
                              oob_score=True,
                              n_jobs=-1,
                              random_state=42,
                              max_depth=5,
                              warm_start=True,
                              bootstrap=True)
model.fit(X_train,y_train)

RandomForestClassifier(max_depth=5, n_estimators=500, n_jobs=-1, oob_score=True,
                       random_state=42, warm_start=True)

In [127]:
y_pred=model.predict(X_test)

In [128]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:",accuracy_score(y_test,y_pred))
print("Classification Report:",classification_report(y_test,y_pred))

Accuracy: 0.8888888888888888
Classification Report:               precision    recall  f1-score   support

           0       0.75      1.00      0.86         6
           1       1.00      0.83      0.91        12

    accuracy                           0.89        18
   macro avg       0.88      0.92      0.88        18
weighted avg       0.92      0.89      0.89        18

